In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import spacy

In [ ]:
product_data = pd.read_excel("Test Data 2.xlsx") 
category_data=pd.read_excel("full unspsc list.xlsx")

In [ ]:
product_data.dropna(subset=['Short Text'],inplace=True)

In [ ]:
nlp = spacy.load("en_core_web_lg")

In [ ]:
def filter_data(text):
    nouns=[]
    doc=nlp(text)
    for token in doc:
        if token.pos_ in ["NOUN","PROPN"]:
            nouns.append(token.text)
    return " ".join(nouns)

In [ ]:
product_data['With Nouns']= product_data['Short Text'].apply(filter_data)

In [ ]:
product_data.head()

In [ ]:
def clean_text(text):
    text=text.lower()
    text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

In [ ]:
product_data['With Nouns'] = product_data['With Nouns'].apply(clean_text)
category_data['Level 4 Description'] = category_data['Level 4 Description'].apply(clean_text)

In [ ]:
vectorizer = CountVectorizer()

In [ ]:
category_vectors = vectorizer.fit_transform(category_data["Level 4 Description"])
product_vectors = vectorizer.transform(product_data["With Nouns"])

In [ ]:
similarity_matrix = cosine_similarity(product_vectors, category_vectors)

In [ ]:
mapped_categories = []

for i in range(len(product_data)):
    similarity_scores= similarity_matrix[i]
    top_indices = similarity_scores.argsort()[-3:][::-1]
    
    top_categories=[]
    for idx in top_indices:
        top_categories.append(category_data['Level 4 Description'][idx])
    top_categories_string=", ".join(top_categories)
    mapped_categories.append(top_categories_string)

product_data['Mapped Category']=mapped_categories

In [ ]:
product_data.to_excel('Products_categ_Mapped.xlsx',index=False)

**Test**

In [ ]:
product_data['Short Text'][51]

In [ ]:
product_data['Mapped Category'][51]